In [1]:
import tensorflow as tf
import numpy as np
from keras.src.metrics.accuracy_metrics import accuracy

print(tf.__version__)

2.16.2


# Architecture du modèle

In [2]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
import os
import pandas as pd

def create_class_dataframe(name: str, folder: str, label: int = 0):
    valid_ext = (".jpg", ".jpeg", ".png", ".bmp")
    files = [f for f in os.listdir(folder) if f.lower().endswith(valid_ext)]
    paths = [os.path.join(folder, f) for f in files]
    df = pd.DataFrame({"path": paths, "label": label})
    print(f"DataFrame créé pour {name} : {len(df)} images")
    return df

def create_mixed_dataset(df_photo, df_other, image_size=(256, 256), batch_size=32, test_ratio=0.2, seed=42):
    # Équilibrage
    n = min(len(df_photo), len(df_other))
    df_mix = pd.concat([
        df_photo.sample(n, random_state=seed),
        df_other.sample(n, random_state=seed)
    ]).sample(frac=1, random_state=seed).reset_index(drop=True)

    train_df, val_df = train_test_split(df_mix, test_size=test_ratio, random_state=seed, stratify=df_mix["label"])

    def df_to_dataset(df):
        path_ds = tf.data.Dataset.from_tensor_slices(df["path"].values)
        label_ds = tf.data.Dataset.from_tensor_slices(df["label"].values)

        def load_image(path):
            img = tf.io.read_file(path)
            img = tf.image.decode_image(img, channels=3, expand_animations=False)
            img = tf.image.resize(img, image_size)
            img = tf.cast(img, tf.float32) / 255.0
            return img

        img_ds = path_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        return tf.data.Dataset.zip((img_ds, label_ds)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    train_ds = df_to_dataset(train_df)
    val_ds = df_to_dataset(val_df)

    print(f"\n Mix créé : {n} images par classe")
    print(f"Train : {len(train_df)} - Val : {len(val_df)}")

    return train_ds, val_ds

# 1. Création des DataFrames intiaux
df_photo = create_class_dataframe("Photo", "./../datasets/Photo", label=1)
df_text = create_class_dataframe("Text", "./../datasets/Text", label=0)
df_sketch = create_class_dataframe("Sketch", "./../datasets/Sketch", label=0)
df_schematics = create_class_dataframe("Schematics", "./../datasets/Schematics", label=0)
df_painting = create_class_dataframe("Painting", "./../datasets/Painting", label=0)

# 2. Création des datasets mixés (photo + autre)
train_text_ds, val_text_ds = create_mixed_dataset(df_photo, df_text)
train_sketch_ds, val_sketch_ds = create_mixed_dataset(df_photo, df_sketch)
train_schematics_ds, val_schematics_ds = create_mixed_dataset(df_photo, df_schematics)
train_painting_ds, val_painting_ds = create_mixed_dataset(df_photo, df_painting)

DataFrame créé pour Photo : 9993 images
DataFrame créé pour Text : 10000 images
DataFrame créé pour Sketch : 1406 images
DataFrame créé pour Schematics : 10000 images
DataFrame créé pour Painting : 10000 images

 Mix créé : 9993 images par classe
Train : 15988 - Val : 3998

 Mix créé : 1406 images par classe
Train : 2249 - Val : 563

 Mix créé : 9993 images par classe
Train : 15988 - Val : 3998

 Mix créé : 9993 images par classe
Train : 15988 - Val : 3998


2025-04-08 10:55:06.290425: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-04-08 10:55:06.290458: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2025-04-08 10:55:06.290462: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2025-04-08 10:55:06.290479: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-08 10:55:06.290488: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
from src.model_loader import ModelLoader

model_loader = ModelLoader(model_name='CNN_hard')
cnn_hard = model_loader.create_model_CNN_hard(show_summary=False)

res_net = model_loader.create_model_resnet50(show_summary=False)

inception = model_loader.create_model_with_inception(show_summary=False)

/Users/tanguydumontier/PycharmProjects/CESI_DS/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
with tf.device("/gpu:0"):
    history = cnn_hard.fit(train_text_ds, epochs=5, verbose=1)

Epoch 1/5


2025-04-08 10:55:45.201528: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


500/500 ━━━━━━━━━━━━━━━━━━━━ 74s 146ms/step - accuracy: 0.9508 - loss: 0.4427
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 72s 144ms/step - accuracy: 0.9977 - loss: 0.0162
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 75s 150ms/step - accuracy: 0.9982 - loss: 0.0088
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 79s 158ms/step - accuracy: 0.9865 - loss: 0.2376
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 79s 158ms/step - accuracy: 0.9919 - loss: 0.1916


In [5]:
with tf.device("/gpu:0"):
    history_res = res_net.fit(train_text_ds, epochs=5,verbose=2)

Epoch 1/5
500/500 - 132s - 264ms/step - accuracy: 0.9534 - loss: 0.2418
Epoch 2/5
500/500 - 133s - 266ms/step - accuracy: 0.9911 - loss: 0.0847
Epoch 3/5
500/500 - 132s - 264ms/step - accuracy: 0.9927 - loss: 0.0582
Epoch 4/5
500/500 - 131s - 262ms/step - accuracy: 0.9927 - loss: 0.0463
Epoch 5/5
500/500 - 130s - 259ms/step - accuracy: 0.9932 - loss: 0.0391


In [ ]:
with tf.device("/gpu:0"):
    history_inception = inception.fit(train_text_ds, epochs=5, verbose=1)

Epoch 1/5
312/500 ━━━━━━━━━━━━━━━━━━━━ 32s 173ms/step - accuracy: 0.9683 - loss: 0.0623

In [6]:
from matplotlib import pyplot as plt

def showTrainingHistory(history: tf.keras.callbacks.History):
    plt.figure()
    plt.plot(history.history['accuracy'])
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper right')
    plt.show()

    plt.figure()
    plt.plot(history.history['loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper right')
    plt.show()

showTrainingHistory(history_res)

NameError: name 'history_res' is not defined

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrix(model, x_test, y_test, class_names=None, normalize=False):
    # Prédictions : on obtient les classes prédites (argmax si probabilités/logits)
    y_pred = model.predict(x_test)
    if y_pred.shape[-1] > 1:
        y_pred = np.argmax(y_pred, axis=1)
    else:
        y_pred = (y_pred > 0.5).astype("int32").flatten()  # Cas binaire

    # Si y_test a la forme (N, 1), on le "flatten"
    y_true = y_test.flatten() if len(y_test.shape) > 1 else y_test

    # Génération de la matrice
    cm = confusion_matrix(y_true, y_pred, normalize='true' if normalize else None)

    # Affichage
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Matrice de confusion" + (" normalisée" if normalize else ""))
    plt.show()

In [5]:
result_loss, result_accuracy = inception.evaluate(val_text_ds, verbose=2)
print(result_loss, result_accuracy)

125/125 - 22s - 175ms/step - accuracy: 0.5018 - loss: 18.9879
18.987884521484375 0.5017508864402771
